# Trabalho Prático - Estrutura de Dados II
## Análise estrutural da malha viária de Cruzeta/RN com OSMnx, NetworkX e Gephi

**Região de estudo:** Cruzeta, Rio Grande do Norte, Brasil.

Este notebook apresenta uma análise estrutural da malha viária de Cruzeta/RN modelada como grafo. Os dados são obtidos a partir do OpenStreetMap por meio da biblioteca OSMnx, processados com NetworkX e exportados para visualização complementar no Gephi.

O objetivo é identificar elementos estruturalmente relevantes da rede urbana, com ênfase em hubs, centralidades, decomposição k-core e diferenças entre a leitura geográfica e a leitura estrutural do grafo.

Além dos artefatos para o Gephi, o notebook gera um mapa interativo em HTML para navegação com zoom, camadas e popups.


## Organização metodológica

O procedimento adotado no notebook segue as etapas solicitadas no enunciado do trabalho:

- obtenção da rede viária com `network_type="drive"`;
- conversão do grafo do OSMnx para uma representação simples e não direcionada, adequada às métricas estruturais;
- cálculo de grau, distribuição de grau, hubs, betweenness centrality, closeness centrality, eigenvector centrality e core number;
- identificação do k-core principal e preparação de atributos para filtros no Gephi;
- produção de tabelas e gráficos quantitativos necessários à interpretação;
- geração de um mapa HTML interativo com zoom, camadas e popups;
- exportação do grafo enriquecido em `.graphml` para visualização geográfica e estrutural no Gephi.


## 1. Instalação das dependências

Esta etapa garante a disponibilidade das bibliotecas utilizadas quando o notebook for executado no Google Colab. Em ambientes locais nos quais as dependências já estejam instaladas, a célula não realiza alterações.


In [ ]:
# A instalação é executada apenas no Google Colab, evitando reinstalações desnecessárias
# quando o notebook for aberto em um ambiente local já configurado.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    import sys

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "osmnx==2.1.0",
        "networkx",
        "pandas",
        "numpy",
        "matplotlib",
        "seaborn",
        "folium",
    ])


## 2. Importações e parâmetros do estudo

Nesta etapa são carregadas as bibliotecas de análise, visualização e manipulação de grafos. Também são definidos a localidade analisada, o tipo de rede viária e os diretórios de saída que receberão tabelas, gráficos e o arquivo destinado ao Gephi.


In [ ]:
from pathlib import Path
import json
import html
import math
import shutil

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
import folium
from folium import plugins

# Padronização visual dos gráficos gerados no notebook.
sns.set_theme(style="whitegrid", context="notebook")

# Parâmetros centrais da coleta no OpenStreetMap.
PLACE = "Cruzeta, Rio Grande do Norte, Brazil"
REGION_LABEL = "Cruzeta/RN, Brasil"
NETWORK_TYPE = "drive"

# Diretórios de saída usados ao final da análise.
OUTPUTS = Path("outputs")
FIGURES = OUTPUTS / "figures"
TABLES = OUTPUTS / "tables"
GEPHI = OUTPUTS / "gephi"
MAPS = OUTPUTS / "maps"

for path in (FIGURES, TABLES, GEPHI, MAPS):
    path.mkdir(parents=True, exist_ok=True)

# O cache reduz chamadas repetidas ao OSMnx quando a mesma consulta é executada novamente.
ox.settings.use_cache = True
ox.settings.log_console = False


## 3. Construção do grafo com OSMnx

A rede viária é extraída do OpenStreetMap com `network_type="drive"`, de modo que o grafo represente vias destinadas ao deslocamento por automóveis. No grafo original do OSMnx, os nós correspondem a interseções ou pontos relevantes da geometria viária, e as arestas correspondem a segmentos de ruas ou rodovias.

Os atributos `x` e `y`, respectivamente longitude e latitude, são preservados porque serão usados posteriormente no Gephi para a visualização geográfica com Geo Layout.


In [ ]:
# Coleta da malha viária diretamente do OpenStreetMap.
G_drive = ox.graph_from_place(PLACE, network_type=NETWORK_TYPE, simplify=True)

print(G_drive)
print(f"Número de nós no MultiDiGraph: {G_drive.number_of_nodes():,}")
print(f"Número de arestas no MultiDiGraph: {G_drive.number_of_edges():,}")
print(f"CRS: {G_drive.graph.get('crs')}")


## 4. Preparação do grafo para análise estrutural

O grafo retornado pelo OSMnx é um `MultiDiGraph`, pois pode conter direção de tráfego e arestas paralelas. Para as métricas estruturais utilizadas neste trabalho, adota-se uma representação simples e não direcionada. Essa escolha torna a análise compatível com o cálculo de grau, k-core, closeness e betweenness em uma interpretação topológica da malha urbana.

Durante a conversão, self-loops são removidos, arestas paralelas são colapsadas usando o menor comprimento disponível e, caso necessário, mantém-se a maior componente conectada. O comprimento das vias (`length`) é preservado para ponderar métricas baseadas em distância.


In [ ]:
def clean_text(value):
    """Normaliza atributos textuais do OSM para exportação em CSV/GraphML."""
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return "; ".join(str(item) for item in value)
    return str(value)


def to_simple_undirected_graph(graph):
    """Converte o MultiDiGraph do OSMnx em grafo simples não direcionado."""
    simple = nx.Graph()

    # Preserva coordenadas e atributos básicos dos nós para análise e Gephi.
    for node, data in graph.nodes(data=True):
        simple.add_node(
            node,
            label=str(node),
            x=float(data["x"]),
            y=float(data["y"]),
            longitude=float(data["x"]),
            latitude=float(data["y"]),
            street_count=int(data.get("street_count", 0)),
        )

    # Remove self-loops e colapsa arestas paralelas mantendo a menor distância.
    for u, v, _key, data in graph.edges(keys=True, data=True):
        if u == v:
            continue

        length = float(data.get("length", 1.0))
        attrs = {
            "length": length,
            "length_m": length,
            "weight": 1.0,
            "name": clean_text(data.get("name")),
            "highway": clean_text(data.get("highway")),
            "ref": clean_text(data.get("ref")),
            "osmid": clean_text(data.get("osmid")),
        }

        if simple.has_edge(u, v):
            if length < simple[u][v]["length"]:
                simple[u][v].update(attrs)
        else:
            simple.add_edge(u, v, **attrs)

    # Mantém a maior componente conectada para evitar distorções em métricas globais.
    if not nx.is_connected(simple):
        largest = max(nx.connected_components(simple), key=len)
        simple = simple.subgraph(largest).copy()

    return simple


G = to_simple_undirected_graph(G_drive)

print(type(G))
print(f"Nós no grafo simples: {G.number_of_nodes():,}")
print(f"Arestas no grafo simples: {G.number_of_edges():,}")
print(f"Grafo conectado? {nx.is_connected(G)}")
print(f"Comprimento total de arestas: {sum(nx.get_edge_attributes(G, 'length').values()) / 1000:,.2f} km")


## 5. Cálculo das métricas estruturais

Esta etapa calcula as métricas obrigatórias do trabalho. O grau identifica conectividade local; a betweenness identifica pontos de intermediação nos menores caminhos; a closeness mede proximidade média em relação ao restante da rede; e o core number indica a participação do nó em subestruturas mais coesas.

As métricas de betweenness e closeness são ponderadas pelo comprimento das vias em metros, o que aproxima a análise topológica de uma interpretação espacial da rede urbana.


In [ ]:
def incident_roads(graph, node, limit=8):
    """Lista nomes e classificações das vias incidentes a um nó."""
    labels = []
    for neighbor in graph.neighbors(node):
        edge = graph[node][neighbor]
        parts = [edge.get("name", ""), edge.get("ref", ""), edge.get("highway", "")]
        label = " / ".join(part for part in parts if part)
        if label:
            labels.append(label)
    return " | ".join(sorted(set(labels))[:limit])


# Métricas locais e globais solicitadas no enunciado.
degree = dict(G.degree())
degree_centrality = nx.degree_centrality(G)
closeness = nx.closeness_centrality(G, distance="length")
betweenness = nx.betweenness_centrality(G, normalized=True, weight="length")
core_number = nx.core_number(G)
eigenvector = nx.eigenvector_centrality(G, max_iter=2000, tol=1e-6)
eccentricity = nx.eccentricity(G)

# O maior core encontrado é adotado como k de referência para o filtro do Gephi.
max_core = max(core_number.values())
selected_k = max_core

# Rankings utilizados para hubs por grau e destaque dos principais nós por betweenness.
ranked_by_degree = sorted(G.nodes, key=lambda n: (degree[n], betweenness[n], str(n)), reverse=True)
top_degree_count = math.ceil(0.10 * G.number_of_nodes())
top_degree_set = set(ranked_by_degree[:top_degree_count])

ranked_by_betweenness = sorted(G.nodes, key=lambda n: (betweenness[n], degree[n], str(n)), reverse=True)
top_betweenness_set = set(ranked_by_betweenness[:10])

# Consolidação das métricas em uma tabela única por nó.
rows = []
for node in G.nodes:
    rows.append(
        {
            "node": node,
            "label": str(node),
            "longitude": G.nodes[node]["x"],
            "latitude": G.nodes[node]["y"],
            "degree": degree[node],
            "degree_centrality": degree_centrality[node],
            "betweenness": betweenness[node],
            "closeness": closeness[node],
            "eigenvector": eigenvector[node],
            "core_number": core_number[node],
            "k_shell": core_number[node],
            "eccentricity": eccentricity[node],
            "top_10pct_degree": int(node in top_degree_set),
            "top_10_betweenness": int(node in top_betweenness_set),
            "selected_k_core": int(core_number[node] >= selected_k),
            "max_core_node": int(core_number[node] == max_core),
            "incident_roads": incident_roads(G, node),
        }
    )

nodes_df = pd.DataFrame(rows)

# Rankings explícitos facilitam ordenação e filtragem no Gephi.
degree_rank = {node: idx + 1 for idx, node in enumerate(ranked_by_degree)}
bet_rank = {node: idx + 1 for idx, node in enumerate(ranked_by_betweenness)}
nodes_df["degree_rank"] = nodes_df["node"].map(degree_rank)
nodes_df["degree_rank_pct"] = nodes_df["degree_rank"] / len(nodes_df)
nodes_df["betweenness_rank"] = nodes_df["node"].map(bet_rank)

# Os atributos calculados são adicionados ao grafo antes da exportação em GraphML.
for row in nodes_df.itertuples(index=False):
    G.nodes[row.node].update(
        {
            "degree": int(row.degree),
            "degree_centrality": float(row.degree_centrality),
            "betweenness": float(row.betweenness),
            "closeness": float(row.closeness),
            "eigenvector": float(row.eigenvector),
            "core_number": int(row.core_number),
            "k_shell": int(row.k_shell),
            "eccentricity": int(row.eccentricity),
            "top_10pct_degree": int(row.top_10pct_degree),
            "top_10_betweenness": int(row.top_10_betweenness),
            "selected_k_core": int(row.selected_k_core),
            "max_core_node": int(row.max_core_node),
            "degree_rank": int(row.degree_rank),
            "degree_rank_pct": float(row.degree_rank_pct),
            "betweenness_rank": int(row.betweenness_rank),
        }
    )

nodes_df.head()


## 6. Caracterização geral da rede

O resumo numérico apresenta uma visão sintética da rede analisada. Esses indicadores descrevem escala, conectividade, densidade, valores de core e extensão do subgrafo k-core selecionado, servindo como base para a interpretação posterior das tabelas e visualizações.


In [ ]:
# Subgrafo k-core usado como referência para filtro estrutural no Gephi.
selected_core_nodes = nodes_df.loc[nodes_df["core_number"] >= selected_k, "node"].tolist()
selected_core = G.subgraph(selected_core_nodes)

# Limiar associado ao top 10% por ranking de grau.
degree_threshold = nodes_df.sort_values(
    ["degree", "betweenness"], ascending=[False, False]
).iloc[top_degree_count - 1]["degree"]

summary = {
    "regiao": REGION_LABEL,
    "consulta_osmnx": PLACE,
    "network_type": NETWORK_TYPE,
    "nos": G.number_of_nodes(),
    "arestas": G.number_of_edges(),
    "conectado": nx.is_connected(G),
    "comprimento_total_km": sum(nx.get_edge_attributes(G, "length").values()) / 1000,
    "densidade": nx.density(G),
    "grau_medio": np.mean(list(degree.values())),
    "grau_maximo": max(degree.values()),
    "quantidade_top_10pct_grau": top_degree_count,
    "limiar_grau_top_10pct_por_ranking": int(degree_threshold),
    "nos_com_grau_maior_ou_igual_ao_limiar": int((nodes_df["degree"] >= degree_threshold).sum()),
    "max_core": max_core,
    "k_escolhido": selected_k,
    "nos_kcore_escolhido": selected_core.number_of_nodes(),
    "arestas_kcore_escolhido": selected_core.number_of_edges(),
    "percentual_nos_kcore_escolhido": selected_core.number_of_nodes() / G.number_of_nodes(),
    "diametro_nao_ponderado": max(eccentricity.values()),
    "raio_nao_ponderado": min(eccentricity.values()),
    "sobreposicao_top10_betweenness_com_top10pct_grau": len(top_degree_set & top_betweenness_set),
}

display(pd.DataFrame(summary.items(), columns=["indicador", "valor"]))


## 7. Tabelas analíticas

As tabelas a seguir reúnem os resultados quantitativos essenciais para responder às questões propostas no trabalho. Elas permitem comparar hubs por grau, nós com maior intermediação, nós mais próximos do restante da rede, distribuição de grau e distribuição por core number.


In [ ]:
columns = [
    "node", "degree", "betweenness", "closeness", "core_number",
    "latitude", "longitude", "incident_roads"
]

# Hubs locais: nós com maior grau. Empates são ordenados por betweenness.
top_hubs_by_degree = nodes_df.sort_values(
    ["degree", "betweenness", "node"], ascending=[False, False, False]
).head(10)

# Pontes estruturais: nós que aparecem em muitos menores caminhos ponderados por comprimento.
top_betweenness = nodes_df.sort_values(
    ["betweenness", "degree", "node"], ascending=[False, False, False]
).head(10)

# Centralidade de proximidade: nós com menor distância média ponderada até os demais.
top_closeness = nodes_df.sort_values(
    ["closeness", "betweenness", "node"], ascending=[False, False, False]
).head(10)

# Distribuições usadas para avaliar homogeneidade e composição estrutural.
degree_distribution = (
    nodes_df.groupby("degree")
    .size()
    .rename("count")
    .reset_index()
    .assign(percent=lambda df: df["count"] / len(nodes_df))
)

core_distribution = (
    nodes_df.groupby("core_number")
    .size()
    .rename("count")
    .reset_index()
    .assign(percent=lambda df: df["count"] / len(nodes_df))
)

metric_summary = (
    nodes_df[["degree", "degree_centrality", "betweenness", "closeness", "core_number"]]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
)

# Tabela de comparação direta entre hubs por grau e nós de maior betweenness.
top10_degree_nodes = set(top_hubs_by_degree["node"])
top10_betweenness_nodes = set(top_betweenness["node"])
comparison_nodes = top10_degree_nodes | top10_betweenness_nodes
hub_betweenness_comparison = (
    nodes_df[nodes_df["node"].isin(comparison_nodes)]
    .assign(
        in_top10_degree=lambda df: df["node"].isin(top10_degree_nodes),
        in_top10_betweenness=lambda df: df["node"].isin(top10_betweenness_nodes),
    )
    .sort_values(
        ["in_top10_betweenness", "in_top10_degree", "betweenness", "degree"],
        ascending=[False, False, False, False],
    )
    [["node", "degree", "betweenness", "closeness", "core_number", "in_top10_degree", "in_top10_betweenness", "incident_roads"]]
)

print("Resumo estatístico das métricas obrigatórias:")
display(metric_summary)

print("Top 10 hubs por grau:")
display(top_hubs_by_degree[columns])

print("Top 10 nós por betweenness:")
display(top_betweenness[columns])

print("Top 10 nós por closeness:")
display(top_closeness[columns])

print("Comparação direta entre top 10 por grau e top 10 por betweenness:")
display(hub_betweenness_comparison)

print("Distribuição de grau:")
display(degree_distribution)

print("Distribuição de core number:")
display(core_distribution)


## 8. Visualizações quantitativas no notebook

Foram mantidos apenas gráficos diretamente associados às questões analíticas do trabalho. A visualização da rede como grafo, tanto em posição geográfica quanto em layout de força, será realizada no Gephi a partir do arquivo `.graphml`.


In [ ]:
# 1) Distribuição de grau e CDF: avalia homogeneidade e concentração de conectividade.
degree_counts = nodes_df["degree"].value_counts().sort_index()
cumulative = degree_counts.cumsum() / degree_counts.sum()

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.bar(degree_counts.index, degree_counts.values, color="#2b8cbe", alpha=0.78, width=0.75, label="Contagem")
ax1.set_xlabel("Grau")
ax1.set_ylabel("Número de nós")

ax2 = ax1.twinx()
ax2.step(cumulative.index, cumulative.values, where="post", color="#d7191c", linewidth=2.3, label="CDF")
ax2.scatter(cumulative.index, cumulative.values, color="#d7191c", s=35, zorder=3)
ax2.set_ylim(0, 1.05)
ax2.set_ylabel("Probabilidade acumulada")

ax1.set_title("Distribuição de grau e CDF", loc="left", fontsize=13, weight="bold")
fig.tight_layout()
fig.savefig(FIGURES / "01_distribuicao_grau_cdf.png", dpi=220)
plt.show()

# 2) Grau versus betweenness: compara conectividade local e intermediação global.
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    nodes_df["degree"],
    nodes_df["betweenness"],
    c=nodes_df["core_number"],
    cmap="viridis",
    s=32,
    alpha=0.68,
    linewidths=0,
)
ax.scatter(
    top_betweenness["degree"],
    top_betweenness["betweenness"],
    s=130,
    facecolors="none",
    edgecolors="#d7191c",
    linewidths=1.8,
    label="Top 10 betweenness",
)
ax.set_xlabel("Grau")
ax.set_ylabel("Betweenness centrality")
ax.set_title("Grau local x intermediação", loc="left", fontsize=13, weight="bold")
fig.colorbar(scatter, ax=ax, label="Core number")
ax.legend(frameon=True)
fig.tight_layout()
fig.savefig(FIGURES / "02_grau_vs_betweenness.png", dpi=220)
plt.show()

# 3) Distribuição de core number: indica o tamanho relativo dos núcleos estruturais.
fig, ax = plt.subplots(figsize=(7, 4.8))
core_counts = nodes_df["core_number"].value_counts().sort_index()
ax.bar(core_counts.index.astype(str), core_counts.values, color="#66c2a5", alpha=0.85)
ax.set_xlabel("Core number")
ax.set_ylabel("Número de nós")
ax.set_title("Distribuição por core number", loc="left", fontsize=13, weight="bold")
for idx, value in enumerate(core_counts.values):
    ax.text(idx, value, f" {value}", va="bottom", ha="center", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES / "03_distribuicao_core.png", dpi=220)
plt.show()


## 9. Mapa geográfico interativo

Esta seção gera uma alternativa complementar às imagens estáticas do Gephi. O mapa é salvo como HTML e permite zoom, deslocamento, ativação de camadas e inspeção de nós por meio de popups.

O objetivo não é substituir o Gephi. O Gephi continua sendo a ferramenta principal para o layout estrutural, ForceAtlas2 e filtros do grafo. O mapa interativo preserva o contexto geográfico do município sem exigir o recorte de uma imagem estática muito grande no README.


In [ ]:
# Mapa geogr?fico interativo complementar.
# Ele usa a geometria original do OSMnx para desenhar as vias e as m?tricas calculadas
# no notebook para destacar os n?s mais relevantes da rede.
MAPS.mkdir(parents=True, exist_ok=True)
map_path = MAPS / "mapa_interativo_rede_viaria_cruzeta.html"


def geometry_to_latlon_paths(geometry):
    """Converte geometrias de arestas em sequ?ncias latitude/longitude para o Folium."""
    if geometry is None or geometry.is_empty:
        return []
    if geometry.geom_type == "LineString":
        return [[(lat, lon) for lon, lat in geometry.coords]]
    if geometry.geom_type == "MultiLineString":
        return [[(lat, lon) for lon, lat in line.coords] for line in geometry.geoms]
    return []


def build_node_popup(row):
    """Monta o conte?do exibido ao clicar em um n? destacado no mapa."""
    roads = str(row.get("incident_roads") or "Sem via identificada no OSM")
    return f"""
    <div style="font-family: Arial, sans-serif; font-size: 12px; line-height: 1.35;">
        <strong>Nó:</strong> {html.escape(str(row["node"]))}<br>
        <strong>Grau:</strong> {int(row["degree"])}<br>
        <strong>Betweenness:</strong> {float(row["betweenness"]):.6f}<br>
        <strong>Closeness:</strong> {float(row["closeness"]):.6f}<br>
        <strong>Core number:</strong> {int(row["core_number"])}<br>
        <strong>Vias incidentes:</strong><br>{html.escape(roads)}
    </div>
    """


def add_node_layer(map_object, data, name, color, radius, show=True):
    """Adiciona uma camada de n?s ao mapa."""
    layer = folium.FeatureGroup(name=name, show=show)
    for _, row in data.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color,
            weight=1.2,
            fill=True,
            fill_color=color,
            fill_opacity=0.78,
            tooltip=f'Nó {row["node"]} | grau {row["degree"]}',
            popup=folium.Popup(build_node_popup(row), max_width=380),
        ).add_to(layer)
    layer.add_to(map_object)
    return layer


center = [nodes_df["latitude"].mean(), nodes_df["longitude"].mean()]
interactive_map = folium.Map(location=center, zoom_start=13, tiles=None, control_scale=True)

folium.TileLayer("CartoDB positron", name="Base clara", control=True).add_to(interactive_map)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", control=True).add_to(interactive_map)

# Camada de arestas: usa a geometria original do OSMnx quando dispon?vel.
edges_layer = folium.FeatureGroup(name="Rede viária", show=True)
try:
    edges_gdf = ox.graph_to_gdfs(G_drive, nodes=False, edges=True, fill_edge_geometry=True)
    for _, edge in edges_gdf.iterrows():
        for coords in geometry_to_latlon_paths(edge.geometry):
            folium.PolyLine(
                coords,
                color="#495057",
                weight=1.15,
                opacity=0.55,
            ).add_to(edges_layer)
except Exception:
    # Fallback: se a convers?o para GeoDataFrame falhar, desenha segmentos retos entre os n?s.
    for u, v, _data in G.edges(data=True):
        coords = [(G.nodes[u]["y"], G.nodes[u]["x"]), (G.nodes[v]["y"], G.nodes[v]["x"])]
        folium.PolyLine(coords, color="#495057", weight=1.15, opacity=0.55).add_to(edges_layer)

edges_layer.add_to(interactive_map)

# Camadas de n?s. As camadas espec?ficas ajudam a responder perguntas da atividade.
add_node_layer(interactive_map, nodes_df, "Todos os nós", "#8c8c8c", radius=2.0, show=True)
add_node_layer(
    interactive_map,
    nodes_df[nodes_df["selected_k_core"] == 1],
    "K-core escolhido",
    "#2ca25f",
    radius=3.0,
    show=False,
)
add_node_layer(
    interactive_map,
    nodes_df[nodes_df["top_10pct_degree"] == 1],
    "Top 10% por grau",
    "#2b8cbe",
    radius=4.5,
    show=False,
)
add_node_layer(
    interactive_map,
    nodes_df[nodes_df["top_10_betweenness"] == 1],
    "Top 10 betweenness",
    "#d7191c",
    radius=6.5,
    show=True,
)

plugins.Fullscreen(position="topright").add_to(interactive_map)
plugins.MiniMap(toggle_display=True, position="bottomright").add_to(interactive_map)
plugins.MeasureControl(position="topleft", primary_length_unit="meters").add_to(interactive_map)
folium.LayerControl(collapsed=False).add_to(interactive_map)

bounds = [
    [nodes_df["latitude"].min(), nodes_df["longitude"].min()],
    [nodes_df["latitude"].max(), nodes_df["longitude"].max()],
]
interactive_map.fit_bounds(bounds)
interactive_map.save(map_path)

print(f"Mapa interativo salvo em: {map_path}")
interactive_map


## 10. Etapa de visualização no Gephi

O notebook concentra a coleta, o cálculo das métricas e a geração dos dados. As visualizações da rede em si devem ser elaboradas no Gephi, conforme solicitado no enunciado. Essa separação evita duplicar visualizações e reserva ao Gephi as análises de layout geográfico e estrutural.

No Gephi, o arquivo `.graphml` deve ser usado para:

- posicionar os nós geograficamente com Geo Layout, utilizando `x` como longitude e `y` como latitude;
- aplicar ForceAtlas2 para a leitura estrutural do grafo;
- dimensionar nós por `degree`;
- colorir nós por `core_number`;
- destacar nós com maior `betweenness`, usando `top_10_betweenness == 1` ou ranking por `betweenness`;
- filtrar o top 10% por grau com `top_10pct_degree == 1`;
- filtrar o k-core escolhido com `selected_k_core == 1` ou `core_number >= k_escolhido`.


## 11. Exportação dos artefatos

Esta etapa salva as tabelas utilizadas na análise, exporta o grafo enriquecido para o formato `.graphml` e gera um arquivo `.zip` com os resultados. O GraphML contém os atributos necessários para visualização, codificação visual e filtragem no Gephi.

Nesta versão do projeto, o pacote `.zip` também inclui o mapa HTML salvo em `outputs/maps/`, desde que a seção anterior tenha sido executada antes da exportação.


In [ ]:
# Exportação das tabelas analíticas em CSV.
nodes_df.to_csv(TABLES / "cruzeta_nodes_metrics.csv", index=False, encoding="utf-8")
degree_distribution.to_csv(TABLES / "cruzeta_degree_distribution.csv", index=False, encoding="utf-8")
core_distribution.to_csv(TABLES / "cruzeta_core_distribution.csv", index=False, encoding="utf-8")
metric_summary.to_csv(TABLES / "cruzeta_metric_summary.csv", index=False, encoding="utf-8")
top_hubs_by_degree.to_csv(TABLES / "cruzeta_top_hubs_by_degree.csv", index=False, encoding="utf-8")
top_betweenness.to_csv(TABLES / "cruzeta_top_betweenness.csv", index=False, encoding="utf-8")
top_closeness.to_csv(TABLES / "cruzeta_top_closeness.csv", index=False, encoding="utf-8")
hub_betweenness_comparison.to_csv(TABLES / "cruzeta_hubs_vs_betweenness.csv", index=False, encoding="utf-8")

# Resumo geral em JSON para consulta rápida.
with open(TABLES / "cruzeta_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

# Arquivo principal para importação no Gephi.
graphml_path = GEPHI / "cruzeta_rn_rede_urbana.graphml"
nx.write_graphml(G, graphml_path)

# Pacote final com figuras, tabelas e GraphML.
zip_path = shutil.make_archive("entrega_cruzeta_rn", "zip", OUTPUTS)

print(f"GraphML para Gephi: {graphml_path}")
print(f"Pacote ZIP com outputs: {zip_path}")
print("Arquivos de saída:")
for path in sorted(OUTPUTS.rglob("*")):
    if path.is_file():
        print(" -", path)


## 12. Procedimento de visualização no Gephi

**Importação**

1. Importar `outputs/gephi/cruzeta_rn_rede_urbana.graphml`.
2. Verificar na aba *Data Laboratory* a presença dos atributos `x`, `y`, `degree`, `betweenness`, `closeness`, `core_number`, `top_10pct_degree` e `selected_k_core`.

**Visualização geográfica**

1. Instalar ou habilitar o plugin **Geo Layout**.
2. Aplicar Geo Layout com longitude = `x` e latitude = `y`.
3. Definir o tamanho dos nós proporcional a `degree`.
4. Definir a cor dos nós a partir de `core_number`.
5. Destacar os nós com `top_10_betweenness == 1`.

**Visualização estrutural**

1. Preservar ou exportar a visualização geográfica antes de alterar o layout.
2. Aplicar **ForceAtlas2**.
3. Manter a mesma codificação visual: tamanho por `degree`, cor por `core_number` e destaque por `betweenness`.
4. Comparar a forma geográfica com a forma estrutural para identificar agrupamentos, pontes e regiões de maior concentração.

**Filtros obrigatórios**

- Top 10% dos nós por grau: `top_10pct_degree == 1`.
- K-core escolhido: `selected_k_core == 1` ou `core_number >= k_escolhido`.
- A justificativa do valor de `k` deve utilizar o resumo numérico do notebook e a distribuição de `core_number`.


## 13. Relação entre evidências e questões analíticas

**1. Os nós com maior grau coincidem com os nós de maior betweenness?**

A comparação deve utilizar a tabela **Comparação direta entre top 10 por grau e top 10 por betweenness** e o gráfico **Grau local x intermediação**. No Gephi, a análise visual deve contrastar o tamanho dos nós, associado a `degree`, com o destaque dos nós de maior `betweenness`.

**2. O núcleo identificado pelo k-core coincide com os principais hubs?**

A resposta deve relacionar a tabela **Top 10 hubs por grau**, a coluna `core_number` e o gráfico **Distribuição por core number**. No Gephi, aplica-se o filtro `selected_k_core == 1` para verificar se os principais hubs permanecem no núcleo filtrado.

**3. O que a betweenness revela que o grau não revela?**

A tabela **Top 10 nós por betweenness** permite identificar nós que funcionam como pontos de passagem, mesmo quando não apresentam o maior grau. A coluna `incident_roads` auxilia a associar esses nós às vias urbanas correspondentes. No Gephi, esses nós devem ser destacados na visualização geográfica.

**4. O que muda entre a posição geográfica real e o layout estrutural?**

O notebook fornece as métricas e o arquivo `.graphml`; a comparação visual é realizada no Gephi. A visualização com Geo Layout preserva a posição real dos nós, enquanto o ForceAtlas2 reorganiza o grafo segundo padrões de conectividade.

**5. Existem regiões críticas para mobilidade urbana na área analisada?**

A identificação dessas regiões deve partir dos nós com maior `betweenness` e das vias listadas em `incident_roads`. No Gephi, a localização desses nós na visualização geográfica permite avaliar se eles conectam setores distintos da rede.

**6. A rede parece homogênea ou apresenta concentração estrutural?**

A avaliação deve combinar o gráfico **Distribuição de grau e CDF**, a tabela **Distribuição de grau**, o resumo das métricas e o gráfico **Distribuição por core number**. No Gephi, observa-se se os maiores nós e os pontos de maior betweenness estão dispersos ou concentrados em poucos corredores.

**7. Os resultados fazem sentido considerando o conhecimento urbano da região escolhida?**

A interpretação deve confrontar as métricas calculadas com o conhecimento sobre Cruzeta/RN. As colunas `incident_roads`, `latitude` e `longitude`, somadas à visualização geográfica no Gephi, permitem verificar se os nós destacados correspondem a vias principais, entradas e saídas da cidade ou áreas centrais.
